# 🔄 Flux de travail d'agent de base avec Microsoft Foundry (Python)

## 📋 Tutoriel d'orchestration de flux de travail

Ce notebook présente les puissantes capacités du **Workflow Builder** du Microsoft Agent Framework. Apprenez à créer des flux de travail d'agents sophistiqués à plusieurs étapes capables de gérer des processus métiers complexes et de coordonner plusieurs opérations d'IA de manière fluide.

> **Note de migration :** Cet exemple faisait auparavant référence aux Modèles GitHub. Les Modèles GitHub sont dépréciés (retrait prévu en juillet 2026), ils utilisent désormais **Microsoft Foundry** via le `FoundryChatClient`, qui cible l'API **Responses** de Azure OpenAI.

## 🎯 Objectifs d'apprentissage

### 🏗️ **Architecture du flux de travail**
- **Workflow Builder** : Concevoir et orchestrer des processus complexes à plusieurs étapes
- **Exécution pilotée par événements** : Gérer les événements et transitions d'état du flux de travail
- **Conception visuelle de flux de travail** : Créer et visualiser les structures de flux de travail
- **Intégration Microsoft Foundry** : Exploiter les modèles d'IA dans les contextes de flux de travail

### 🔄 **Orchestration des processus**
- **Opérations séquentielles** : Enchaîner plusieurs tâches d'agent dans un ordre logique
- **Logique conditionnelle** : Implémenter des points de décision et des branches dans les flux de travail
- **Gestion des erreurs** : Récupération robuste et résilience du flux de travail
- **Gestion d'état** : Suivre et gérer l'état d'exécution du flux de travail

### 📊 **Modèles de flux de travail d'entreprise**
- **Automatisation des processus métier** : Automatiser des flux de travail organisationnels complexes
- **Coordination multi-agent** : Coordonner plusieurs agents spécialisés
- **Exécution évolutive** : Concevoir des flux de travail pour des opérations à l'échelle de l'entreprise
- **Surveillance et observabilité** : Suivre la performance et les résultats des flux de travail

## ⚙️ Prérequis et configuration

### 📦 **Dépendances requises**

Installez le Agent Framework avec les capacités de flux de travail :

```bash
pip install agent-framework -U
```

### 🔑 **Configuration Microsoft Foundry**

Connectez-vous avec Azure CLI (`az login`) pour que `AzureCliCredential` puisse s'authentifier, puis configurez les détails de votre projet Microsoft Foundry.

**Configuration de l'environnement (fichier .env) :**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

### 🏢 **Cas d'usage en entreprise**

**Exemples de processus métier :**
- **Intégration client** : Flux de travail de vérification et configuration en plusieurs étapes
- **Pipeline de contenu** : Création, révision et publication automatisées de contenu
- **Traitement de données** : Flux ETL avec transformation assistée par IA
- **Assurance qualité** : Tests et validations automatisés

**Avantages des flux de travail :**
- 🎯 **Fiabilité** : Exécution déterministe avec récupération d'erreurs
- 📈 **Scalabilité** : Gérer l'automatisation de processus à fort volume
- 🔍 **Observabilité** : Traçabilité complète et monitoring
- 🔧 **Maintenabilité** : Conception visuelle et composants modulaires

## 🎨 Modèles de conception de flux de travail

### Structure basique du flux de travail
```mermaid
graph TD
    A[Début] --> B[Tâche Agent 1]
    B --> C{Point de décision}
    C -->|Succès| D[Tâche Agent 2]
    C -->|Échec| E[Gestionnaire d'erreur]
    D --> F[Fin]
    E --> F
```

**Composants clés :**
- **WorkflowBuilder** : Moteur principal d'orchestration
- **WorkflowEvent** : Gestion des événements et communication
- **WorkflowViz** : Représentation visuelle et débogage des flux de travail

Construisons votre premier flux de travail intelligent ! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
# Core components for building sophisticated agent workflows
from agent_framework import WorkflowBuilder, WorkflowEvent, WorkflowViz
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:
# 📦 Import Environment and System Utilities
# Essential libraries for configuration and environment management

import os                      # 🔧 Environment variable access
from dotenv import load_dotenv # 📁 Secure configuration loading

In [ ]:
# 🔧 Initialize Environment Configuration
# Load Microsoft Foundry project settings from .env file
load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
REVIEWER_NAME = "Concierge"
REVIEWER_INSTRUCTIONS = """
    You are an are hotel concierge who has opinions about providing the most local and authentic experiences for travelers.
    The goal is to determine if the front desk travel agent has recommended the best non-touristy experience for a traveler.
    If so, state that it is approved.
    If not, provide insight on how to refine the recommendation without using a specific example. 
    """

In [ ]:
FRONTDESK_NAME = "FrontDesk"
FRONTDESK_INSTRUCTIONS = """
    You are a Front Desk Travel Agent with ten years of experience and are known for brevity as you deal with many customers.
    The goal is to provide the best activities and locations for a traveler to visit.
    Only provide a single recommendation per response.
    You're laser focused on the goal at hand.
    Don't waste time with chit chat.
    Consider suggestions when refining an idea.
    """

In [ ]:
reviewer_agent = provider.as_agent(
    name=REVIEWER_NAME,
    instructions=REVIEWER_INSTRUCTIONS,
)

front_desk_agent = provider.as_agent(
    name=FRONTDESK_NAME,
    instructions=FRONTDESK_INSTRUCTIONS,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=front_desk_agent)
    .add_edge(front_desk_agent, reviewer_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")


In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The reviewer (last stage) is the only output here.
events = await workflow.run("I would like to go to Paris.")
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
